# **Data retrieval pipeline**




[Get players via their PUUIDs]

↓

[Get match IDs]

↓

[Get match data]

↓

[Extract lane matchups]

↓

[Store dataset]

---

In [2]:
import requests
import time
import pandas as pd
import os

from tqdm.notebook import tqdm
from dotenv import load_dotenv

load_dotenv()

True

Chose region and match region here.

In [ ]:
API_KEY = os.getenv("MY_API_KEY")
HEADERS = {"X-Riot-Token": API_KEY}
REGION = "euw1"
MATCH_REGION = "europe"

### **Extrating players and match IDs**

Retrieve emerald I player data. But each page on the Riot side only returns around 200 players, so we need to loop through multiple pages to get more players via the `get_many_emerald_players` function. The `get_match_ids` function is just where we specify the player ID and the number of matches we want from them, and it returns a list of match IDs. Finally, the `get_match` function takes in a match ID and returns the match data in JSON format.

NOTE: if not emerald players, then just change the URL to whatever rank or division:

```python
url = f"https://{REGION}.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/EMERALD/I?page={page}"
```

For instance, `EMERALD/I` can be changed to for example `GOLD/II` or `PLATINUM/IV`.

In [ ]:
def get_emerald_1_players(page=1):
    url = f"https://{REGION}.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/EMERALD/I?page={page}"
    
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        print("Error:", response.status_code)
        return []
    
    data = response.json()
    
    return [entry["puuid"] for entry in data]

def get_many_emerald_players(pages=3):
    players = []
    
    for page in range(1, pages + 1):
        batch = get_emerald_1_players(page)
        
        if not batch:
            break
        
        players.extend(batch)
        
        time.sleep(1.2) # rate limit
    
    return players

def get_match_ids(puuid, count=5):
    url = f"https://{MATCH_REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?count={count}"
    
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        print("Error fetching match IDs:", response.status_code)
        return []
    
    return response.json()

def get_match(match_id):
    url = f"https://{MATCH_REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        print("Error fetching match:", response.status_code)
        return None
    
    return response.json()

<div class="alert alert-info">
CHOOSE HOW MANY PLAYERS AND HOW MANY MATCHES PER PLAYER HERE. BE CAREFUL THOUGH, THIS CAN TAKE A LONG TIME IF TOO MANY IS CHOSEN.
</div>

Choosing 5 pages hence expecting around 1000 players, and 3 matches per player, so expecting around 3000 match IDs.

In [ ]:
players = get_many_emerald_players(pages=5)

match_ids = get_match_ids(players[0], count=3)

len(players)

Testing for match data retrieval.

In [ ]:
match = get_match(match_ids[0])

print(match)

### **Extracting lane matchups**

In [ ]:
def extract_matchups(match_json):
    results = []
    
    participants = match_json["info"]["participants"]
    
    for p in participants:
        role = p["teamPosition"]
        
        if role == "":
            continue
        
        if role == "UTILITY":
            role = "SUPPORT"
        
        opponent = next(
            (x for x in participants 
             if x["teamPosition"] == p["teamPosition"] and x["teamId"] != p["teamId"]),
            None
        )
        
        if opponent:
            results.append({
                "champion": p["championName"],
                "opponent": opponent["championName"],
                "role": role,
                "win": int(p["win"])
            })
    
    return results

In [ ]:
match_data = extract_matchups(match)
match_data[:10]

### **Creating the dataset**

In [ ]:
all_data = []
seen_matches = set()

pbar = tqdm(players, desc="Processing players")

for puuid in pbar:
    
    pbar.set_postfix({
        "matches": len(seen_matches),
        "rows": len(all_data)
    })
    
    match_ids = get_match_ids(puuid, count=3)
    time.sleep(1.5)
    
    for match_id in match_ids:
        
        if match_id in seen_matches:
            continue
        
        seen_matches.add(match_id)
        
        match = get_match(match_id)
        
        if match:
            extracted = extract_matchups(match)
            all_data.extend(extracted)
        
        time.sleep(1.5)

For 1000 players and 3 matches per player, the code ran for 121 minutes and 40 seconds.

In [ ]:
df = pd.DataFrame(all_data)

### **Winwrate calculation**

This is just for testing.

In [ ]:
matchups = (
    df.groupby(["champion", "opponent", "role"])
    .agg(
        winrate=("win", "mean"),
        games=("win", "count")
    )
    .reset_index()
)

# Find all matchups with at least 5 games to ensure some level of reliability in the winrate
matchups = matchups[matchups["games"] >= 10]

In [ ]:
# Top 5 matchups by winrate
matchups.sort_values("winrate", ascending=False).head()

In [ ]:
# debugging champion names
[m for m in matchups["champion"] if "twisted" in m.lower()]

### **Saving the dataset**

In [ ]:
df.to_csv("data/matchups_raw_1000.csv", index=False)
matchups.to_csv("data/matchups_aggregated_1000.csv", index=False)